# 34. The Hazard & IMDG Segregation Problem

## Tier 3: The Advanced Algorithm (Particle Swarm Optimization Implementation)

### Goal
Implement Particle Swarm Optimization (PSO) for the IMDG segregation problem, treating each particle as a complete container placement solution with particles exploring the solution space through velocity updates guided by segregation constraint satisfaction.

### Key assumptions
- Each particle represents a complete assignment of containers to positions
- Particles move through solution space guided by personal and global best solutions
- Fitness function combines segregation violations with placement efficiency metrics
- Swarm intelligence emerges from particle interactions and information sharing

### Approach (step-by-step)
1. Initialize particle swarm with random container assignments
2. Define fitness function combining violations, distance penalties, and hold utilization
3. Update particle velocities using PSO equations with inertia and acceleration coefficients
4. Apply position updates and handle constraint violations through repair mechanisms
5. Track personal best and global best solutions throughout iterations
6. Analyze convergence and compare with baseline methods

### What to look for in the results
- Convergence behavior showing fitness improvement over iterations
- Reduced segregation violations compared to random initialization
- Balance between exploration (diversity) and exploitation (convergence)
- Solution quality comparable to or better than heuristic methods

### Concrete example (from the source)
PSO with 30 particles for 50 iterations on a 12-container, 20-position problem with mixed IMDG classes. Initial random placement yields 8 segregation violations with fitness score 8247. After PSO optimization, the algorithm converges to a solution with 0 violations and fitness score 156, demonstrating a 98% improvement.

In [ ]:
# Import required packages for PSO implementation
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple, Optional
import random
import time
from dataclasses import dataclass
import warnings
warnings.filterwarnings('ignore')

# Set up visualization style
plt.style.use('default')
sns.set_palette("husl")

In [ ]:
# Define data structures for PSO implementation
@dataclass
class Container:
    """Represents a container with dangerous goods"""
    id: str
    imdg_class: str
    weight: float = 20.0

@dataclass
class Position:
    """Represents a stowage position on the vessel"""
    id: str
    hold: int
    x: float
    y: float
    z: float

@dataclass
class SegregationRequirement:
    """Represents segregation requirements between container classes"""
    class1: str
    class2: str
    requirement: str
    min_distance: float
    different_hold: bool = False

@dataclass
class Particle:
    """Represents a particle in the PSO algorithm"""
    position: np.ndarray  # Container assignments
    velocity: np.ndarray  # Velocity vector
    fitness: float
    best_position: np.ndarray  # Personal best position
    best_fitness: float  # Personal best fitness

@dataclass
class PSOConfig:
    """PSO algorithm configuration parameters"""
    swarm_size: int = 30
    max_iterations: int = 50
    inertia_weight: float = 0.7
    cognitive_coeff: float = 1.5  # c1 - personal best influence
    social_coeff: float = 1.5    # c2 - global best influence
    velocity_clamp: float = 2.0
    max_velocity: float = 4.0

In [ ]:
# Create the PSO example problem
def create_pso_example():
    """Create a 12-container, 20-position problem for PSO demonstration"""
    
    # Define 12 containers with mixed IMDG classes
    imdg_classes = ["1.1", "1.4S", "3", "3", "8", "8.1", "2.1", "2.3", "5.1", "6.1", "9", "9"]
    containers = []
    
    for i, imdg_class in enumerate(imdg_classes):
        containers.append(Container(f"C{i+1}", imdg_class, 20.0))
    
    # Define 20 positions across 4 holds (5 positions per hold)
    positions = []
    for hold in range(1, 5):  # Holds 1, 2, 3, 4
        for pos_idx in range(5):  # 5 positions per hold
            x = pos_idx * 3  # 3m spacing
            positions.append(Position(f"H{hold}_P{pos_idx+1}", hold, x, 0, 0))
    
    # Define comprehensive segregation requirements
    segregation_requirements = [
        # Explosives restrictions (most restrictive)
        SegregationRequirement("1.1", "3", "Away From", 6.0),
        SegregationRequirement("1.1", "8", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "2.1", "Away From", 6.0),
        SegregationRequirement("1.1", "2.3", "Away From", 6.0),
        SegregationRequirement("1.1", "5.1", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "6.1", "Separated From", 0.0, True),
        SegregationRequirement("1.1", "9", "Away From", 3.0),
        
        # Class 1.4S restrictions
        SegregationRequirement("1.4S", "3", "Away From", 3.0),
        SegregationRequirement("1.4S", "8", "Separated From", 0.0, True),
        SegregationRequirement("1.4S", "2.1", "Away From", 3.0),
        SegregationRequirement("1.4S", "2.3", "Away From", 3.0),
        
        # Flammable liquids restrictions
        SegregationRequirement("3", "8", "Away From", 3.0),
        SegregationRequirement("3", "2.1", "No restriction", 0.0),
        SegregationRequirement("3", "2.3", "Away From", 3.0),
        SegregationRequirement("3", "5.1", "Away From", 3.0),
        SegregationRequirement("3", "6.1", "Away From", 3.0),
        SegregationRequirement("3", "9", "No restriction", 0.0),
        
        # Corrosives restrictions
        SegregationRequirement("8", "2.1", "Away From", 3.0),
        SegregationRequirement("8", "2.3", "Away From", 3.0),
        SegregationRequirement("8", "5.1", "Away From", 3.0),
        SegregationRequirement("8", "6.1", "Away From", 3.0),
        SegregationRequirement("8", "9", "Away From", 3.0),
        
        # Gases restrictions
        SegregationRequirement("2.1", "2.3", "Away From", 3.0),
        SegregationRequirement("2.1", "5.1", "Away From", 3.0),
        SegregationRequirement("2.1", "6.1", "Away From", 3.0),
        SegregationRequirement("2.1", "9", "No restriction", 0.0),
        
        # Organic peroxides and toxic materials
        SegregationRequirement("5.1", "6.1", "Away From", 3.0),
        SegregationRequirement("5.1", "9", "Away From", 3.0),
        SegregationRequirement("6.1", "9", "Away From", 3.0),
    ]
    
    return containers, positions, segregation_requirements

# Create the PSO example
containers, positions, seg_reqs = create_pso_example()

print(f"PSO Problem Instance:")
print(f"  Containers: {len(containers)}")
print(f"  Positions: {len(positions)}")
print(f"  Segregation requirements: {len(seg_reqs)}")

print(f"\nContainer Classes:")
class_counts = {}
for c in containers:
    class_counts[c.imdg_class] = class_counts.get(c.imdg_class, 0) + 1
for imdg_class, count in sorted(class_counts.items()):
    print(f"  Class {imdg_class}: {count} containers")

print(f"\nHold Distribution:")
hold_counts = {}
for p in positions:
    hold_counts[p.hold] = hold_counts.get(p.hold, 0) + 1
for hold in sorted(hold_counts.keys()):
    print(f"  Hold {hold}: {hold_counts[hold]} positions")

In [ ]:
# Utility functions for PSO
def calculate_distance_matrix(positions: List[Position]) -> Dict[Tuple[str, str], float]:
    """Calculate Euclidean distances between all position pairs"""
    distances = {}
    for i, pos1 in enumerate(positions):
        for j, pos2 in enumerate(positions):
            if i != j:
                dist = np.sqrt((pos1.x - pos2.x)**2 + (pos1.y - pos2.y)**2 + (pos1.z - pos2.z)**2)
                distances[(pos1.id, pos2.id)] = dist
    return distances

def check_segregation_compliance(container1: Container, container2: Container, 
                                pos1: Position, pos2: Position, 
                                requirements: List[SegregationRequirement],
                                distances: Dict[Tuple[str, str], float]) -> bool:
    """Check if two containers in given positions satisfy segregation requirements"""
    
    # Find applicable requirement
    applicable_req = None
    for req in requirements:
        if ((req.class1 == container1.imdg_class and req.class2 == container2.imdg_class) or
            (req.class1 == container2.imdg_class and req.class2 == container1.imdg_class)):
            applicable_req = req
            break
    
    if applicable_req is None or applicable_req.requirement == "No restriction":
        return True
    
    # Check distance requirement
    distance = distances.get((pos1.id, pos2.id), 0)
    if distance < applicable_req.min_distance:
        return False
    
    # Check different hold requirement
    if applicable_req.different_hold and pos1.hold == pos2.hold:
        return False
    
    return True

# Calculate distance matrix
distance_matrix = calculate_distance_matrix(positions)

In [ ]:
# PSO fitness function and particle operations
def calculate_fitness(position_vector: np.ndarray,
                    containers: List[Container],
                    positions: List[Position],
                    requirements: List[SegregationRequirement],
                    distances: Dict[Tuple[str, str], float],
                    w1: float = 1000.0,  # Violation penalty weight
                    w2: float = 1.0,     # Distance penalty weight
                    w3: float = 10.0) -> float:  # Hold utilization weight
    """Calculate fitness for a particle position (lower is better)"""
    
    # Decode position vector to container assignments
    assignments = {}
    for i, pos_idx in enumerate(position_vector):
        if 0 <= pos_idx < len(positions):
            assignments[containers[i].id] = positions[int(pos_idx)]
    
    # Component 1: Segregation violations (highest penalty)
    total_violations = 0
    violation_penalty = 0
    
    for i, container1 in enumerate(containers):
        for j, container2 in enumerate(containers):
            if i < j:
                pos1 = assignments.get(container1.id)
                pos2 = assignments.get(container2.id)
                
                if pos1 and pos2 and pos1.id != pos2.id:
                    if not check_segregation_compliance(container1, container2, pos1, pos2, requirements, distances):
                        total_violations += 1
                        
                        # Calculate violation severity
                        for req in requirements:
                            if ((req.class1 == container1.imdg_class and req.class2 == container2.imdg_class) or
                                (req.class1 == container2.imdg_class and req.class2 == container1.imdg_class)):
                                if req.different_hold and pos1.hold == pos2.hold:
                                    violation_penalty += 100  # Same hold violation is severe
                                elif req.min_distance > 0:
                                    distance = distances.get((pos1.id, pos2.id), 0)
                                    if distance < req.min_distance:
                                        violation_penalty += (req.min_distance - distance) * 10
                                break
    
    # Component 2: Distance inefficiency (prefer good spacing)
    total_distance_penalty = 0
    for i, container1 in enumerate(containers):
        for j, container2 in enumerate(containers):
            if i < j:
                pos1 = assignments.get(container1.id)
                pos2 = assignments.get(container2.id)
                
                if pos1 and pos2:
                    # Penalize containers that are too close (even if compliant)
                    distance = distances.get((pos1.id, pos2.id), 0)
                    if distance < 6.0:  # Less than 6m is inefficient
                        total_distance_penalty += (6.0 - distance)
    
    # Component 3: Hold utilization balance (prefer balanced distribution)
    hold_usage = {}
    for assignment in assignments.values():
        hold_usage[assignment.hold] = hold_usage.get(assignment.hold, 0) + 1
    
    # Calculate imbalance penalty
    ideal_per_hold = len(containers) / 4  # 4 holds
    hold_imbalance = sum(abs(usage - ideal_per_hold) for usage in hold_usage.values())
    
    # Combined fitness (lower is better)
    fitness = w1 * violation_penalty + w2 * total_distance_penalty + w3 * hold_imbalance
    
    return fitness

def initialize_particle(swarm_size: int, num_containers: int, num_positions: int) -> List[Particle]:
    """Initialize particle swarm with random positions and velocities"""
    
    particles = []
    
    for _ in range(swarm_size):
        # Random position (each container assigned to random position)
        position = np.random.uniform(0, num_positions - 1, num_containers)
        
        # Random velocity (small initial velocities)
        velocity = np.random.uniform(-1, 1, num_containers)
        
        # Create particle
        particle = Particle(
            position=position,
            velocity=velocity,
            fitness=float('inf'),
            best_position=position.copy(),
            best_fitness=float('inf')
        )
        
        particles.append(particle)
    
    return particles

In [ ]:
# PSO algorithm implementation
def particle_swarm_optimization(containers: List[Container],
                               positions: List[Position],
                               requirements: List[SegregationRequirement],
                               distances: Dict[Tuple[str, str], float],
                               config: PSOConfig) -> Dict:
    """Implement PSO algorithm for IMDG segregation"""
    
    start_time = time.time()
    
    num_containers = len(containers)
    num_positions = len(positions)
    
    # Initialize swarm
    particles = initialize_particle(config.swarm_size, num_containers, num_positions)
    
    # Initialize global best
    global_best_position = None
    global_best_fitness = float('inf')
    
    # Track convergence
    convergence_history = []
    diversity_history = []
    
    print(f"PSO Algorithm - Swarm Size: {config.swarm_size}, Max Iterations: {config.max_iterations}")
    print("=" * 70)
    
    for iteration in range(config.max_iterations):
        # Evaluate fitness for all particles
        for particle in particles:
            particle.fitness = calculate_fitness(
                particle.position, containers, positions, requirements, distances
            )
            
            # Update personal best
            if particle.fitness < particle.best_fitness:
                particle.best_fitness = particle.fitness
                particle.best_position = particle.position.copy()
        
        # Update global best
        for particle in particles:
            if particle.fitness < global_best_fitness:
                global_best_fitness = particle.fitness
                global_best_position = particle.position.copy()
        
        # Record convergence
        convergence_history.append(global_best_fitness)
        
        # Calculate swarm diversity
        positions_matrix = np.array([p.position for p in particles])
        diversity = np.mean(np.std(positions_matrix, axis=0))
        diversity_history.append(diversity)
        
        # Update particles
        for particle in particles:
            # Generate random coefficients
            r1 = np.random.random(num_containers)
            r2 = np.random.random(num_containers)
            
            # Update velocity (PSO equation)
            cognitive_component = config.cognitive_coeff * r1 * (particle.best_position - particle.position)
            social_component = config.social_coeff * r2 * (global_best_position - particle.position)
            
            particle.velocity = (config.inertia_weight * particle.velocity + 
                              cognitive_component + social_component)
            
            # Velocity clamping
            particle.velocity = np.clip(particle.velocity, -config.max_velocity, config.max_velocity)
            
            # Update position
            particle.position = particle.position + particle.velocity
            
            # Position clamping (ensure within bounds)
            particle.position = np.clip(particle.position, 0, num_positions - 1)
            
            # Repair mechanism: ensure integer positions and no duplicates
            particle.position = np.round(particle.position)
            
            # Handle duplicate positions (simple repair)
            unique_positions, counts = np.unique(particle.position.astype(int), return_counts=True)
            duplicates = unique_positions[counts > 1]
            
            for dup_pos in duplicates:
                # Find indices of duplicates
                dup_indices = np.where(particle.position.astype(int) == dup_pos)[0]
                # Keep first, reassign others
                for idx in dup_indices[1:]:
                    available_positions = set(range(num_positions)) - set(particle.position.astype(int))
                    if available_positions:
                        particle.position[idx] = random.choice(list(available_positions))
        
        # Progress reporting
        if (iteration + 1) % 10 == 0 or iteration == 0:
            print(f"Iteration {iteration + 1:3d}: Best Fitness = {global_best_fitness:.2f}, Diversity = {diversity:.2f}")
    
    execution_time = time.time() - start_time
    
    # Decode final solution
    final_assignments = {}
    for i, pos_idx in enumerate(global_best_position):
        if 0 <= pos_idx < len(positions):
            final_assignments[containers[i].id] = positions[int(pos_idx)]
    
    # Calculate final solution quality
    final_violations = 0
    for i, container1 in enumerate(containers):
        for j, container2 in enumerate(containers):
            if i < j:
                pos1 = final_assignments.get(container1.id)
                pos2 = final_assignments.get(container2.id)
                if pos1 and pos2 and pos1.id != pos2.id:
                    if not check_segregation_compliance(container1, container2, pos1, pos2, requirements, distances):
                        final_violations += 1
    
    return {
        'best_position': global_best_position,
        'best_fitness': global_best_fitness,
        'assignments': final_assignments,
        'violations': final_violations,
        'convergence_history': convergence_history,
        'diversity_history': diversity_history,
        'execution_time': execution_time,
        'particles': particles
    }

In [ ]:
# Run PSO algorithm
pso_config = PSOConfig(
    swarm_size=30,
    max_iterations=50,
    inertia_weight=0.7,
    cognitive_coeff=1.5,
    social_coeff=1.5,
    velocity_clamp=2.0,
    max_velocity=4.0
)

print("IMDG Segregation - Particle Swarm Optimization")
print("=" * 60)
print(f"Problem: {len(containers)} containers, {len(positions)} positions")
print(f"PSO Configuration: {pso_config.swarm_size} particles, {pso_config.max_iterations} iterations")
print()

# Run PSO
pso_result = particle_swarm_optimization(containers, positions, seg_reqs, distance_matrix, pso_config)

print("\n" + "=" * 60)
print("PSO OPTIMIZATION RESULTS:")
print("=" * 60)
print(f"Best Fitness: {pso_result['best_fitness']:.2f}")
print(f"Segregation Violations: {pso_result['violations']}")
print(f"Execution Time: {pso_result['execution_time']:.4f} seconds")
print(f"Containers Successfully Placed: {len(pso_result['assignments'])}/{len(containers)}")

# Calculate improvement over random baseline
random_fitness = calculate_fitness(
    np.random.uniform(0, len(positions)-1, len(containers)),
    containers, positions, seg_reqs, distance_matrix
)
improvement = ((random_fitness - pso_result['best_fitness']) / random_fitness) * 100

print(f"\nBaseline Comparison:")
print(f"Random Solution Fitness: {random_fitness:.2f}")
print(f"PSO Solution Fitness: {pso_result['best_fitness']:.2f}")
print(f"Improvement: {improvement:.1f}%")

In [ ]:
# Visualize PSO convergence and solution
def visualize_pso_results(pso_result: Dict, containers: List[Container], positions: List[Position]):
    """Create comprehensive visualization of PSO results"""
    
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    
    # Plot 1: Convergence history
    ax1 = axes[0, 0]
    iterations = range(1, len(pso_result['convergence_history']) + 1)
    ax1.plot(iterations, pso_result['convergence_history'], 'b-', linewidth=2)
    ax1.set_xlabel('Iteration')
    ax1.set_ylabel('Best Fitness')
    ax1.set_title('PSO Convergence History')
    ax1.grid(True, alpha=0.3)
    ax1.set_yscale('log')  # Log scale for better visualization
    
    # Plot 2: Swarm diversity
    ax2 = axes[0, 1]
    ax2.plot(iterations, pso_result['diversity_history'], 'g-', linewidth=2)
    ax2.set_xlabel('Iteration')
    ax2.set_ylabel('Swarm Diversity')
    ax2.set_title('Swarm Diversity Over Time')
    ax2.grid(True, alpha=0.3)
    
    # Plot 3: Fitness distribution (final swarm)
    ax3 = axes[0, 2]
    final_fitnesses = [p.fitness for p in pso_result['particles']]
    ax3.hist(final_fitnesses, bins=15, alpha=0.7, color='skyblue', edgecolor='black')
    ax3.axvline(pso_result['best_fitness'], color='red', linestyle='--', linewidth=2, label=f'Best: {pso_result["best_fitness"]:.2f}')
    ax3.set_xlabel('Fitness Value')
    ax3.set_ylabel('Number of Particles')
    ax3.set_title('Final Swarm Fitness Distribution')
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    # Plot 4-6: Container placement by hold
    holds = {}
    for hold_num in range(1, 5):
        holds[hold_num] = [p for p in positions if p.hold == hold_num]
    
    def draw_hold_layout(ax, hold_positions, title):
        ax.set_title(title, fontsize=12, fontweight='bold')
        ax.set_xlabel('Horizontal Position (m)')
        ax.set_ylabel('Vertical Position (m)')
        ax.grid(True, alpha=0.3)
        ax.set_xlim(-1, 13)
        ax.set_ylim(-1, 3)
        
        # Color mapping for IMDG classes
        class_colors = {
            '1.1': 'darkred', '1.4S': 'red', '3': 'orange', '8': 'purple',
            '8.1': 'darkviolet', '2.1': 'green', '2.3': 'darkgreen',
            '5.1': 'brown', '6.1': 'olive', '9': 'blue'
        }
        
        for pos in hold_positions:
            # Find container at this position
            container_info = None
            for container_id, assigned_pos in pso_result['assignments'].items():
                if assigned_pos.id == pos.id:
                    container_info = next(c for c in containers if c.id == container_id)
                    break
            
            if container_info:
                color = class_colors.get(container_info.imdg_class, 'gray')
                ax.scatter(pos.x, pos.y, s=300, c=color, marker='s', alpha=0.8, edgecolors='black')
                ax.text(pos.x, pos.y, f'{container_info.id}\n{container_info.imdg_class}', 
                        ha='center', va='center', fontsize=7, fontweight='bold')
            else:
                ax.scatter(pos.x, pos.y, s=300, c='lightgray', marker='s', alpha=0.3, edgecolors='black')
                ax.text(pos.x, pos.y, 'Empty', ha='center', va='center', fontsize=6)
    
    # Draw holds
    for i, hold_num in enumerate(range(1, 4)):
        ax = axes[1, i]
        draw_hold_layout(ax, holds[hold_num], f'Hold {hold_num} - Final Placement')
    
    plt.suptitle('IMDG Segregation - PSO Optimization Results', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Additional analysis: Violation details
    fig, ax = plt.subplots(figsize=(12, 8))
    ax.axis('tight')
    ax.axis('off')
    
    # Analyze final solution
    violation_details = []
    for i, container1 in enumerate(containers):
        for j, container2 in enumerate(containers):
            if i < j:
                pos1 = pso_result['assignments'].get(container1.id)
                pos2 = pso_result['assignments'].get(container2.id)
                if pos1 and pos2 and pos1.id != pos2.id:
                    is_compliant = check_segregation_compliance(container1, container2, pos1, pos2, seg_reqs, distance_matrix)
                    distance = distance_matrix.get((pos1.id, pos2.id), 0)
                    
                    # Find requirement
                    requirement = "No restriction"
                    min_dist = 0
                    for req in seg_reqs:
                        if ((req.class1 == container1.imdg_class and req.class2 == container2.imdg_class) or
                            (req.class1 == container2.imdg_class and req.class2 == container1.imdg_class)):
                            requirement = req.requirement
                            min_dist = req.min_distance
                            break
                    
                    violation_details.append({
                        'container1': container1.id,
                        'class1': container1.imdg_class,
                        'container2': container2.id,
                        'class2': container2.imdg_class,
                        'requirement': requirement,
                        'min_distance': min_dist,
                        'actual_distance': distance,
                        'compliant': is_compliant
                    })
    
    # Create compliance table
    table_data = [['Pair', 'Classes', 'Requirement', 'Positions', 'Distance', 'Status']]
    colors = [['lightgray', 'lightgray', 'lightgray', 'lightgray', 'lightgray', 'lightgray']]
    
    for detail in violation_details[:20]:  # Show first 20 pairs
        status = "✓" if detail['compliant'] else "✗"
        status_color = 'lightgreen' if detail['compliant'] else 'lightcoral'
        
        table_data.append([
            f"{detail['container1']} - {detail['container2']}",
            f"{detail['class1']} - {detail['class2']}",
            detail['requirement'],
            f"{pso_result['assignments'][detail['container1']].id} - {pso_result['assignments'][detail['container2']].id}",
            f"{detail['actual_distance']:.1f}m",
            status
        ])
        colors.append(['white', 'white', 'white', 'white', 'white', status_color])
    
    table = ax.table(cellText=table_data, cellColours=colors, 
                    cellLoc='center', loc='center', colWidths=[0.12, 0.1, 0.15, 0.25, 0.1, 0.08])
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.8)
    
    # Style header row
    for i in range(6):
        table[(0, i)].set_facecolor('#4472C4')
        table[(0, i)].set_text_props(weight='bold', color='white')
    
    plt.title('PSO Solution - Compliance Analysis', fontsize=14, fontweight='bold', pad=20)
    plt.show()

# Visualize PSO results
visualize_pso_results(pso_result, containers, positions)

In [ ]:
# Parameter sensitivity analysis
def parameter_sensitivity_analysis():
    """Analyze PSO performance with different parameter settings"""
    
    # Test different parameter combinations
    parameter_sets = [
        {'inertia_weight': 0.9, 'cognitive_coeff': 2.0, 'social_coeff': 2.0, 'name': 'High Exploration'},
        {'inertia_weight': 0.7, 'cognitive_coeff': 1.5, 'social_coeff': 1.5, 'name': 'Balanced'},
        {'inertia_weight': 0.5, 'cognitive_coeff': 1.0, 'social_coeff': 2.5, 'name': 'Social Focus'},
        {'inertia_weight': 0.4, 'cognitive_coeff': 2.5, 'social_coeff': 1.0, 'name': 'Cognitive Focus'},
    ]
    
    results = []
    
    print("Parameter Sensitivity Analysis:")
    print("=" * 50)
    
    for params in parameter_sets:
        print(f"\nTesting: {params['name']}")
        
        # Create config with test parameters
        test_config = PSOConfig(
            swarm_size=20,  # Reduced for faster testing
            max_iterations=30,  # Reduced for faster testing
            inertia_weight=params['inertia_weight'],
            cognitive_coeff=params['cognitive_coeff'],
            social_coeff=params['social_coeff']
        )
        
        # Run PSO
        test_result = particle_swarm_optimization(containers, positions, seg_reqs, distance_matrix, test_config)
        
        # Calculate convergence rate (improvement in first 10 iterations)
        if len(test_result['convergence_history']) >= 10:
            initial_fitness = test_result['convergence_history'][0]
            early_fitness = test_result['convergence_history'][9]
            convergence_rate = (initial_fitness - early_fitness) / initial_fitness * 100
        else:
            convergence_rate = 0
        
        results.append({
            'name': params['name'],
            'final_fitness': test_result['best_fitness'],
            'violations': test_result['violations'],
            'convergence_rate': convergence_rate,
            'execution_time': test_result['execution_time'],
            'final_diversity': test_result['diversity_history'][-1] if test_result['diversity_history'] else 0
        })
        
        print(f"  Final Fitness: {test_result['best_fitness']:.2f}")
        print(f"  Violations: {test_result['violations']}")
        print(f"  Convergence Rate: {convergence_rate:.1f}%")
    
    # Create comparison visualization
    fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(14, 10))
    
    names = [r['name'] for r in results]
    
    # Final fitness comparison
    fitnesses = [r['final_fitness'] for r in results]
    bars1 = ax1.bar(names, fitnesses, color='skyblue', alpha=0.7)
    ax1.set_ylabel('Final Fitness')
    ax1.set_title('Final Solution Quality')
    ax1.grid(True, alpha=0.3, axis='y')
    plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')
    
    for bar, f in zip(bars1, fitnesses):
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(fitnesses)*0.01, 
                f'{f:.1f}', ha='center', va='bottom', fontweight='bold')
    
    # Violations comparison
    violations = [r['violations'] for r in results]
    colors = ['green' if v == 0 else 'orange' if v <= 2 else 'red' for v in violations]
    bars2 = ax2.bar(names, violations, color=colors, alpha=0.7)
    ax2.set_ylabel('Number of Violations')
    ax2.set_title('Segregation Violations')
    ax2.grid(True, alpha=0.3, axis='y')
    plt.setp(ax2.get_xticklabels(), rotation=45, ha='right')
    
    for bar, v in zip(bars2, violations):
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1, 
                str(v), ha='center', va='bottom', fontweight='bold')
    
    # Convergence rate comparison
    conv_rates = [r['convergence_rate'] for r in results]
    bars3 = ax3.bar(names, conv_rates, color='lightgreen', alpha=0.7)
    ax3.set_ylabel('Convergence Rate (%)')
    ax3.set_title('Early Convergence Speed')
    ax3.grid(True, alpha=0.3, axis='y')
    plt.setp(ax3.get_xticklabels(), rotation=45, ha='right')
    
    for bar, cr in zip(bars3, conv_rates):
        ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(conv_rates)*0.01, 
                f'{cr:.1f}%', ha='center', va='bottom', fontweight='bold')
    
    # Final diversity comparison
    diversities = [r['final_diversity'] for r in results]
    bars4 = ax4.bar(names, diversities, color='orange', alpha=0.7)
    ax4.set_ylabel('Final Swarm Diversity')
    ax4.set_title('Solution Diversity')
    ax4.grid(True, alpha=0.3, axis='y')
    plt.setp(ax4.get_xticklabels(), rotation=45, ha='right')
    
    for bar, d in zip(bars4, diversities):
        ax4.text(bar.get_x() + bar.get_width()/2, bar.get_height() + max(diversities)*0.01, 
                f'{d:.2f}', ha='center', va='bottom', fontweight='bold')
    
    plt.suptitle('PSO Parameter Sensitivity Analysis', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print("\n" + "=" * 50)
    print("PARAMETER SENSITIVITY SUMMARY:")
    print("=" * 50)
    for r in results:
        print(f"\n{r['name']}:")
        print(f"  Final Fitness: {r['final_fitness']:.2f}")
        print(f"  Violations: {r['violations']}")
        print(f"  Convergence Rate: {r['convergence_rate']:.1f}%")
        print(f"  Final Diversity: {r['final_diversity']:.2f}")

# Run parameter sensitivity analysis
parameter_sensitivity_analysis()

### Why this Tier exists vs earlier Tiers
This Tier 3 PSO approach addresses limitations of both exact and heuristic methods:
- **Global optimization capability** - Can escape local optima that trap heuristics
- **Balance between exploration and exploitation** - Systematic search vs greedy placement
- **Population-based search** - Multiple solution candidates explored simultaneously
- **Adaptive search behavior** - Swarm intelligence emerges from particle interactions

### Pros / Cons vs Tier 1 (MIP) and Tier 2 (Heuristic)
**Advantages over both Tiers 1 & 2:**
- Better solution quality than simple heuristics through global search
- Faster than MIP for large instances while maintaining good solution quality
- Can find near-optimal solutions where heuristics get stuck in local optima
- Parallelizable nature allows for efficient implementation
- Adaptive behavior adjusts search based on problem landscape

**Disadvantages vs Tier 1:**
- No optimality guarantees (unlike MIP)
- Parameter tuning required for good performance
- More complex to implement and understand than heuristics
- Stochastic nature means results vary between runs

**Disadvantages vs Tier 2:**
- Higher computational cost than simple heuristics
- More complex parameter tuning
- Less predictable behavior than deterministic heuristics

### When to use this Tier
- **Medium to large problems** where heuristics may get stuck in local optima
- **When near-optimal solutions are needed** but MIP is too slow
- **Complex constraint landscapes** with many local optima
- **When multiple good solutions are valuable** (swarm provides diversity)
- **Research and development** where algorithm performance is critical
- **Problems with moderate computational budgets** requiring quality solutions